# Mobile Price Prediction - Exploratory Data Analysis

**Analyzing mobile phone specifications to understand what features determine price range classification.**

## Project Overview

This project explores a mobile phone dataset to understand the relationship between phone specifications (RAM, battery, camera, etc.) and price range categories. Rather than predicting exact prices, we analyze which hardware features most strongly correlate with higher price tiers.

## Learning Objectives

1. Analyze multi-class classification data through EDA
2. Identify key features that drive price segmentation
3. Visualize relationships between technical specifications and price tiers
4. Apply correlation analysis and statistical tests to categorical targets
5. Understand feature importance without building ML models

## Business / Research Problem

Phone manufacturers and retailers need to understand which specifications justify premium pricing. This analysis helps:
- **Manufacturers** prioritize R&D spending on features that move phones to higher price tiers
- **Consumers** understand what they pay for at each price level
- **Retailers** position products effectively

**Key question:** *Which phone specifications are the strongest indicators of price range, and how do features interact across tiers?*

## Why This Analysis Matters

The mobile phone market is highly competitive. Understanding the feature-price relationship enables data-driven product positioning and helps consumers make informed purchasing decisions.

## Dataset Overview

Key features include: `battery_power`, `blue` (bluetooth), `clock_speed`, `dual_sim`, `fc` (front camera MP), `four_g`, `int_memory`, `m_dep` (depth cm), `mobile_wt`, `n_cores`, `pc` (primary camera MP), `px_height`, `px_width`, `ram`, `sc_h` (screen height), `sc_w` (screen width), `talk_time`, `three_g`, `touch_screen`, `wifi`, `price_range` (0-3, target).

**Size:** 2,000 records, 21 features

## Dataset Source & License Notes

- **Source:** [Kaggle - Mobile Price Classification](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification)
- **License:** Open (Educational use)
- **Note:** Synthetic dataset designed for classification practice

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'iabhishekofficial/mobile-price-classification'
SIGNIFICANCE_LEVEL = 0.05
PRICE_LABELS = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Very High'}
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

# Use train.csv which has the price_range target
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print(f'Available files: {csv_files}')
df = pd.read_csv(os.path.join(path, 'train.csv'))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
df.head()

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
if df.isnull().sum().sum() == 0:
    print('No missing values found.')
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nTarget distribution:\n{df["price_range"].value_counts().sort_index()}')

In [ ]:
df.describe()

## Data Cleaning

Verify data quality and create useful derived features.

In [ ]:
df = df.drop_duplicates()
print(f'Rows after dedup: {len(df)}')

# Create readable price label
df['price_label'] = df['price_range'].map(PRICE_LABELS)

# Total pixels
df['total_pixels'] = df['px_height'] * df['px_width']

# Screen area
df['screen_area'] = df['sc_h'] * df['sc_w']

print('Derived features created: price_label, total_pixels, screen_area')

## Exploratory Data Analysis

In [ ]:
print('=== Target Distribution ===')
for pr, label in PRICE_LABELS.items():
    count = len(df[df['price_range'] == pr])
    print(f'  {label} ({pr}): {count} ({count/len(df)*100:.1f}%)')

print(f'\n=== RAM by Price Range ===')
print(df.groupby('price_range')['ram'].agg(['mean', 'median', 'std']).round(0))

print(f'\n=== Battery Power by Price Range ===')
print(df.groupby('price_range')['battery_power'].agg(['mean', 'median']).round(0))

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(df['ram'], bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('RAM Distribution')
axes[0, 0].set_xlabel('RAM (MB)')

axes[0, 1].hist(df['battery_power'], bins=30, color='coral', edgecolor='white')
axes[0, 1].set_title('Battery Power Distribution')
axes[0, 1].set_xlabel('Battery (mAh)')

axes[0, 2].hist(df['int_memory'], bins=30, color='mediumseagreen', edgecolor='white')
axes[0, 2].set_title('Internal Memory Distribution')
axes[0, 2].set_xlabel('Memory (GB)')

axes[1, 0].hist(df['pc'], bins=20, color='mediumpurple', edgecolor='white')
axes[1, 0].set_title('Primary Camera (MP)')

axes[1, 1].hist(df['mobile_wt'], bins=20, color='orange', edgecolor='white')
axes[1, 1].set_title('Mobile Weight (g)')

df['price_label'].value_counts().reindex(['Low', 'Medium', 'High', 'Very High']).plot(
    kind='bar', ax=axes[1, 2], color=sns.color_palette('Set2', 4), edgecolor='white')
axes[1, 2].set_title('Price Range Distribution')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='price_range', y='ram', ax=axes[0], palette='viridis')
axes[0].set_title('RAM by Price Range')
axes[0].set_xticklabels(['Low', 'Med', 'High', 'V.High'])

sns.boxplot(data=df, x='price_range', y='battery_power', ax=axes[1], palette='viridis')
axes[1].set_title('Battery by Price Range')
axes[1].set_xticklabels(['Low', 'Med', 'High', 'V.High'])

sns.boxplot(data=df, x='price_range', y='total_pixels', ax=axes[2], palette='viridis')
axes[2].set_title('Total Pixels by Price Range')
axes[2].set_xticklabels(['Low', 'Med', 'High', 'V.High'])

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: RAM vs Battery colored by price range
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for pr in sorted(df['price_range'].unique()):
    subset = df[df['price_range'] == pr]
    axes[0].scatter(subset['ram'], subset['battery_power'], alpha=0.4, s=15,
                    label=PRICE_LABELS[pr])
axes[0].set_xlabel('RAM (MB)')
axes[0].set_ylabel('Battery Power (mAh)')
axes[0].set_title('RAM vs Battery by Price Range')
axes[0].legend()

for pr in sorted(df['price_range'].unique()):
    subset = df[df['price_range'] == pr]
    axes[1].scatter(subset['ram'], subset['total_pixels'], alpha=0.4, s=15,
                    label=PRICE_LABELS[pr])
axes[1].set_xlabel('RAM (MB)')
axes[1].set_ylabel('Total Pixels')
axes[1].set_title('RAM vs Screen Resolution by Price Range')
axes[1].legend()

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [ ]:
# Feature means by price range
key_features = ['ram', 'battery_power', 'px_height', 'px_width', 'int_memory', 'pc', 'fc', 'mobile_wt']
feature_means = df.groupby('price_range')[key_features].mean().round(1)
feature_means.index = [PRICE_LABELS[i] for i in feature_means.index]
print('=== Feature Means by Price Range ===')
print(feature_means)

# Binary features by price range
binary_features = ['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']
binary_means = df.groupby('price_range')[binary_features].mean().round(3)
binary_means.index = [PRICE_LABELS[i] for i in binary_means.index]
print('\n=== Binary Feature Proportions by Price Range ===')
print(binary_means)

## Statistical Checks / Hypothesis Testing

In [ ]:
# Correlation with price_range
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('price_range')
correlations = df[numeric_cols].corrwith(df['price_range']).abs().sort_values(ascending=False)
print('=== Correlation with Price Range (absolute) ===')
print(correlations.round(4).head(10))

# ANOVA for top features
print('\n=== ANOVA Tests ===')
for feat in ['ram', 'battery_power', 'px_height', 'px_width']:
    groups = [g[feat].values for _, g in df.groupby('price_range')]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f'{feat}: F={f_stat:.2f}, p={p_val:.2e} ({"Significant" if p_val < SIGNIFICANCE_LEVEL else "Not significant"})')

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
corr_matrix = df[list(key_features) + ['price_range']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [ ]:
# Top 10 most correlated features bar chart
fig, ax = plt.subplots(figsize=(10, 6))
correlations.head(10).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 10 Features Correlated with Price Range')
ax.set_xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots for top features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, feat in enumerate(['ram', 'battery_power', 'px_height', 'px_width']):
    ax = axes[i // 2, i % 2]
    sns.violinplot(data=df, x='price_range', y=feat, ax=ax, palette='viridis')
    ax.set_title(f'{feat} by Price Range')
    ax.set_xticklabels(['Low', 'Med', 'High', 'V.High'])
plt.tight_layout()
plt.show()

## Key Findings

1. **RAM is the strongest predictor** of price range — near-perfect linear separation
2. **Battery power and screen resolution** are the next most important features
3. **Binary features** (wifi, bluetooth, 4G) show surprisingly weak correlation with price range
4. **Camera megapixels** have low correlation — not a strong price differentiator in this dataset
5. **Phone weight** does not strongly predict price range
6. **Classes are perfectly balanced** (500 each) — this is a synthetic dataset designed for classification

## Limitations

1. **Synthetic data:** The dataset is artificially generated — real-world relationships may differ
2. **No brand information:** Brand is a major real-world price factor not captured here
3. **Price ranges, not prices:** We only have ordinal categories, not actual dollar values
4. **No software/OS info:** Android vs iOS is a major price factor missing
5. **No temporal data:** Phone specs evolve rapidly — this is a snapshot

## Recommendations / Next Steps

1. Build a classification model (Random Forest, SVM) to predict price_range
2. Use RAM bands to create a simple rule-based classifier as baseline
3. Apply dimensionality reduction (PCA) given the high feature count

### How to Extend This Analysis
- Scrape real phone specs and prices to compare with this synthetic data
- Build a feature importance comparison across different ML models
- Create an interactive dashboard for phone comparison

## Common Mistakes

1. **Treating price_range as continuous:** It's ordinal — use appropriate metrics and tests
2. **Ignoring the synthetic nature:** Drawing real-world business conclusions from this data is risky
3. **Feature scaling oversight:** Features have very different ranges — always scale before modeling
4. **Not checking for multicollinearity:** px_height and px_width are correlated — consider combining
5. **Ignoring RAM dominance:** If RAM alone achieves high accuracy, your model may be learning a trivial pattern

## Mini Challenge / Exercises

1. Create a RAM threshold classifier: can you achieve >80% accuracy with simple rules?
2. Which feature pair gives the best 2D scatter plot separation of price ranges?
3. Calculate the chi-square test for independence between `four_g` and `price_range`
4. Create a parallel coordinates plot for the top 5 features
5. What's the average battery-to-weight ratio for each price range?

In [ ]:
# Space for exercise solutions
# Exercise 1: Simple RAM threshold classifier
# Exercise 2: Try combinations with scatter plots
# Exercise 3: from scipy.stats import chi2_contingency
# Exercise 4: pd.plotting.parallel_coordinates()
# Exercise 5: df['battery_wt_ratio'] = df['battery_power'] / df['mobile_wt']

## Final Summary / Key Takeaways

- **RAM dominates price classification** — it's the single most predictive feature by a wide margin
- **Battery and screen resolution** are secondary but significant predictors
- **Binary connectivity features** (wifi, bluetooth, 4G) barely distinguish price tiers
- **This is synthetic data** — results demonstrate analytical technique but not real market dynamics
- **For ML modeling**, RAM + battery_power + pixel dimensions would form a strong minimal feature set

This analysis shows how systematic EDA can identify the most important features before any modeling begins.